[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/00_cmgdb_intro.ipynb)

# A short CMGDB primer

[CMGDB](https://github.com/bernardorivas/CMGDB) turns a box map into a
Morse graph: recurrent regions, their Conley indices, and the order of
possible transitions. This small planar Leslie example uses CMGDB
directly; the four paper notebooks load the same objects for learned
latent maps.

The corner-based box map below is a numerical illustration, not an
interval-arithmetic certificate. The paper likewise distinguishes its
sampled numerical checks from mathematically certified hypotheses.


In [ ]:
# Colab keeps a checkout so the artifact manifest remains available.
import os
import shutil
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/latent_dynamics")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "paper",
             "https://github.com/begelb/latent_dynamics.git", str(root)],
            check=True,
        )
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cmgdb==1.3.3+fork.3",
         "--find-links", "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.3"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    os.chdir(root)


## Map and box map

The Leslie map advances two population classes. CMGDB asks for a
rectangular image for each input rectangle; here it is estimated from
the four corners.


In [ ]:
import math
import CMGDB

def f(x):
    return [(20*x[0] + 20*x[1]) * math.exp(-0.1*(x[0] + x[1])), 0.7*x[0]]

def box_map(rect):
    a, b, c, d = rect
    images = [f([x, y]) for x in (a, c) for y in (b, d)]
    return [min(z[0] for z in images), min(z[1] for z in images),
            max(z[0] for z in images), max(z[1] for z in images)]


## Morse decomposition

At subdivision 17 the grid resolves the example's two attracting
recurrent objects. A minimal node has no outgoing edge in the Morse
graph.


In [ ]:
model = CMGDB.Model(17, 17, 17, 10_000, [-0.001, -0.001], [90.0, 70.0], box_map)
graph, _ = CMGDB.ComputeMorseGraph(model)
minimal = [v for v in graph.vertices() if not graph.adjacencies(v)]
print(f"{graph.num_vertices()} Morse sets; {len(minimal)} minimal: {minimal}")


In [ ]:
CMGDB.PlotMorseGraph(graph)
CMGDB.PlotMorseSets(graph)
